# Olist Delivery Delay Risk

This notebook is the first exploratory pass for the project.

## Questions
1. How common are late deliveries in the delivered-order subset?
2. Which purchase-time patterns seem associated with late deliveries?
3. Which features look promising before training a first model?

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.io import load_olist_tables
from src.data.prepare import build_modeling_frame
from src.features.engineering import build_model_dataset

sns.set_theme(style="whitegrid")

In [ ]:
data_dir = ROOT / "data" / "raw"
tables = load_olist_tables(data_dir)
modeling_frame = build_modeling_frame(tables)
dataset = build_model_dataset(modeling_frame)

print(f"Rows in modeling frame: {len(modeling_frame):,}")
print(f"Rows in model dataset: {len(dataset):,}")
print(f"Late-delivery rate: {dataset['is_late'].mean():.2%}")
dataset.head()

## Plot 1: Target balance
Use this to confirm the class imbalance before picking metrics.

In [ ]:
ax = sns.countplot(data=dataset, x="is_late")
ax.set(title="Late delivery target balance", xlabel="is_late", ylabel="Order count")
plt.show()

## Plot 2: Late rate by customer state
Limit to the most common states so the chart stays readable.

In [ ]:
top_states = dataset['customer_state'].value_counts().head(10).index
state_rates = (
    dataset.loc[dataset['customer_state'].isin(top_states)]
    .groupby('customer_state', as_index=False)['is_late']
    .mean()
    .sort_values('is_late', ascending=False)
)

ax = sns.barplot(data=state_rates, x='customer_state', y='is_late')
ax.set(title='Late-delivery rate by customer state', xlabel='Customer state', ylabel='Late rate')
plt.show()

## Plot 3: Estimated delivery window vs. target
This checks whether tighter promised windows are associated with more delay risk.

In [ ]:
ax = sns.boxplot(data=dataset, x='is_late', y='estimated_delivery_days')
ax.set(title='Estimated delivery window by target', xlabel='is_late', ylabel='Estimated delivery days')
plt.show()

## Plot 4: Payment type and late rate
This can surface simple operational or customer-behavior patterns.

In [ ]:
payment_rates = (
    dataset.groupby('payment_type_mode', as_index=False)['is_late']
    .mean()
    .sort_values('is_late', ascending=False)
)

ax = sns.barplot(data=payment_rates, x='payment_type_mode', y='is_late')
ax.set(title='Late-delivery rate by payment type', xlabel='Payment type', ylabel='Late rate')
plt.xticks(rotation=30)
plt.show()

## Plot 5: Price distribution by target
This is a quick way to see whether larger baskets behave differently.

In [ ]:
ax = sns.boxplot(data=dataset, x='is_late', y='total_price')
ax.set(title='Total price by target', xlabel='is_late', ylabel='Total price')
plt.show()

## Notes to write after the first pass
- What is the late-delivery base rate?
- Which segments look riskier?
- Which features should definitely stay for the first model?
- Which features should be excluded because of leakage or low trust?